# NeuroMappers — Exploration Notebook

**Project:** DBS Electrode Precise Localization in the Brain  
**Course:** Biomedical Image Analysis (BMIA), University of Santiago de Compostela  
**Team:** Samad Ullah (USC) and Catarina Souto (University of Porto) — supervised by Prof. João Paulo Cunha

---

## Sprint 1 deliverables

1. **Data inspection** — verify MRI and CT format, dimensions, voxel spacing, orientation, and compatibility.
2. **Data loading** — implement functions to load NIfTI MRI and CT volumes into Python for downstream processing.

The library code lives in `src/`. This notebook demonstrates the functions against the Lead-Tutor dataset.

## Dataset

**Lead-Tutor** — Madan et al., *Aperture Neuro* (2025), DOI [10.52294/001c.129658](https://doi.org/10.52294/001c.129658). Ten patients bilaterally implanted with DBS electrodes at Brigham & Women's Hospital.

The OSF archive contains **raw native-space NIfTI only** (31 files total) organised as BIDS under `rawdata/sub-XXXXX/ses-{preop,postop}/anat/`. No bundled registration, no MNI-normalized images, and no expert electrode coordinates ship with the download.

## Setup

Make `src/` importable and load the Sprint 1 helpers.

In [ ]:
# ============================================================
# Project setup + imports for all sprints
# ============================================================
import sys
from pathlib import Path


def find_project_root(marker: str = "requirements.txt") -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not locate project root (no {marker} found above {here})")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# --- Common scientific stack ---------------------------------
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Sprint 1: data inspection -------------------------------
from src.config import LEADTUTOR_DIR, PATIENT_IDS, PROCESSED_DIR
from src.load_and_inspect import (
    load_volume,
    inspect_volume,
    inspect_many,
    find_nifti_files,
)

# --- Sprint 2: preprocessing ---------------------------------
import ants
from src.preprocessing import (
    find_subject_files,
    reorient_to_ras,
    resample_isotropic,
    clip_ct,
    normalize_mri_zscore,
    crop_to_content,
    quality_check,
    preprocess_subject,
    preprocess_all,
    PREPROCESSING_QC_CSV,
)


# --- Sprint 2: registration ---------------------------------
from src.registration import (
    register_subject,
    register_all,
    preprocessed_path,
    REGISTRATION_LOG_CSV,
)

# --- Sanity prints -------------------------------------------
print(f"Project root    : {project_root}")
print(f"Lead-Tutor root : {LEADTUTOR_DIR}")
print(f"Dataset exists  : {LEADTUTOR_DIR.exists()}")
print(f"Processed dir   : {PROCESSED_DIR}")
print(f"QC table dest   : {PREPROCESSING_QC_CSV}")
print(f"Reg log dest    : {REGISTRATION_LOG_CSV}")
print(f"antspyx version : {ants.__version__}")


## 1. Discover NIfTI files

In [ ]:
raw_root = LEADTUTOR_DIR / "rawdata"
nifti_paths = find_nifti_files(raw_root)

print(f"Found {len(nifti_paths)} NIfTI files across {len(PATIENT_IDS)} subjects\n")
for p in nifti_paths[:6]:
    print(p.relative_to(LEADTUTOR_DIR))
print("...")

## 2. Inspect all volumes — format, dimensions, spacing, orientation

Runs `inspect_volume()` against every NIfTI, collects metadata into a DataFrame, and saves the result to `results/sprint1_data_inspection.csv`.

In [ ]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

df = inspect_many(nifti_paths)
df["subject"] = df["path"].str.extract(r"sub-(\d+)")
df["session"] = df["path"].str.extract(r"ses-(preop|postop)")
df = df[[
    "subject", "session", "modality_guess", "format", "shape",
    "voxel_spacing_mm", "orientation", "dtype",
    "min_intensity", "max_intensity", "nan_count",
]]

output_csv = project_root / "results" / "sprint1_data_inspection.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"Saved inspection summary to {output_csv}\n")
df

## 3. Key findings

In [ ]:
print("Unique formats:", df["format"].unique())
print("\nModalities found:", sorted(df["modality_guess"].unique()))
print("\nModality counts:")
print(df["modality_guess"].value_counts().to_string())
print("\nUnique orientations across dataset:", sorted({tuple(o) for o in df["orientation"]}))
print("\nFiles containing NaN voxels:")
nan_rows = df[df["nan_count"] > 0][["subject", "session", "modality_guess", "shape", "nan_count"]]
print(nan_rows.to_string(index=False) if not nan_rows.empty else "  none")

### Observations

- **Format:** all 31 volumes are NIfTI-1 (`.nii.gz`).
- **Orientations vary:** four different axis-code triplets appear across subjects. A reorientation step to a common reference frame will be required before MRI–CT registration.
- **Voxel-spacing heterogeneity:** CTs are ~0.4–0.5 mm in-plane with 0.5–2.0 mm slice thickness; T1w usually isotropic ~1 mm; T2w anisotropic with 1.2–3.0 mm slices.
- **Intensity ranges:** CT values sit in a consistent Hounsfield range `[-1024, 3071]`; MRI intensities vary widely between subjects (no inter-subject normalization yet).
- **Discrepancies with the published dataset description:**
  - sub-39468 is listed in the PDF table as having FLAIR; the actual file is **SWI** (Susceptibility-Weighted Imaging).
  - sub-39468's post-operative CT contains ~7.45 M **NaN voxels** (~21% of the volume) — data-quality issue to raise with the supervisor.
- **MRI vs CT — main technical differences:**
  - CT is always 512×512 axial; MRI is typically 256×256 (about 4× fewer in-plane voxels).
  - CT has the finer in-plane resolution (~0.45 mm vs ~1 mm) but MRI is often isotropic.
  - CT intensity semantics are standardised (Hounsfield Units); MRI intensities are subject- and scanner-dependent.

## 4. Load and visualize a sample MRI / CT pair

Demonstrates the `load_volume()` function from `src.load_and_inspect` returning a NumPy array ready for downstream processing, together with the NIfTI affine matrix.

In [ ]:
sample_subject = "15454"
mri_path = raw_root / f"sub-{sample_subject}" / "ses-preop"  / "anat" / f"sub-{sample_subject}_ses-preop_acq-iso_T1w.nii.gz"
ct_path  = raw_root / f"sub-{sample_subject}" / "ses-postop" / "anat" / f"sub-{sample_subject}_ses-postop_CT.nii.gz"

mri_data, mri_affine, mri_img = load_volume(mri_path)
ct_data,  ct_affine,  ct_img  = load_volume(ct_path)

print(f"MRI T1w  shape={mri_data.shape}  spacing={tuple(round(z, 3) for z in mri_img.header.get_zooms()[:3])} mm")
print(f"CT       shape={ct_data.shape}  spacing={tuple(round(z, 3) for z in ct_img.header.get_zooms()[:3])} mm")

mri_mid = mri_data.shape[2] // 2
ct_mid  = ct_data.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.rot90(mri_data[:, :, mri_mid]), cmap="gray")
axes[0].set_title(f"sub-{sample_subject} · pre-op T1w · axial slice {mri_mid}")
axes[0].axis("off")

axes[1].imshow(np.rot90(ct_data[:, :, ct_mid]), cmap="gray", vmin=-100, vmax=300)
axes[1].set_title(f"sub-{sample_subject} · post-op CT · axial slice {ct_mid}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Sprint 1 status — acceptance criteria

**Task 1 — Data inspection**

- [x] MRI file format identified (NIfTI-1)
- [x] CT file format identified (NIfTI-1)
- [x] Image dimensions recorded for both datasets
- [x] Voxel spacing recorded for both datasets
- [x] Orientation / affine information checked
- [x] Main differences between MRI and CT documented
- [x] Short summary produced (`results/sprint1_data_inspection.csv` + observations above)

**Task 2 — MRI and CT loading**

- [x] MRI loads correctly (`load_volume()` in `src.load_and_inspect`)
- [x] CT loads correctly (same function)
- [x] Files can be read without errors
- [x] Volumes available as NumPy arrays with affine matrices, ready for visualization and processing

**Open questions for Prof. Cunha (to raise in class):** registration scope, source of expert electrode coordinates, handling of sub-39468 NaN issue, patient/target cohort scope — see Jira comment.

---

# Sprint 2 — Task 1: Preprocessing

This section turns the **raw, heterogeneous** Lead-Tutor volumes into **registration-ready** images.
We do this in five fixed steps for every NIfTI file:

1. **Reorient to RAS+** — the 31 raw files use 4 different orientations (LAS, RAS, PSR, RSA).
2. **Resample to 1 mm isotropic** — CT is fine in-plane (~0.4 mm) but coarse through-plane; MRI is anisotropic too.
3. **Intensity normalization** — HU clip `[-1024, 3500]` for CT, foreground z-score for MRI.
4. **Crop to content** — drop the air margins around the head.
5. **Per-volume QC** — record shape, spacing, orientation, finite fraction, foreground fraction, flags.

Library code lives in `src/preprocessing.py`. Outputs are saved to `data/processed/sub-XXXXX/...` and the QC table to `results/sprint2_preprocessing_qc.csv`.


## 1. Per-step preprocessing on sub-15454

We walk through the pipeline manually on one subject so the class can see what each step does.


In [ ]:
SAMPLE_ID = "15454"

files = find_subject_files(SAMPLE_ID)
print(f"sub-{SAMPLE_ID} files found:")
for mod, p in files.items():
    print(f"  {mod:5s}  {p.name}")


In [ ]:
ct_raw  = ants.image_read(str(files["CT"]))
mri_raw = ants.image_read(str(files["T1w"]))

def header(label, img):
    arr = img.numpy()
    nan_n = int(np.isnan(arr).sum())
    print(f"{label:18s} orient={img.orientation}  shape={arr.shape}  "
          f"spacing={tuple(round(s,2) for s in img.spacing)}  "
          f"min={float(np.nanmin(arr)):.1f}  max={float(np.nanmax(arr)):.1f}  nan={nan_n}")

header("CT  raw",  ct_raw)
header("T1w raw",  mri_raw)


In [ ]:
# Step 1+2: reorient to RAS, resample to 1mm iso
ct_std  = resample_isotropic(reorient_to_ras(ct_raw))
mri_std = resample_isotropic(reorient_to_ras(mri_raw))

# Step 3: intensity normalization
ct_norm  = clip_ct(ct_std)
mri_norm = normalize_mri_zscore(mri_std)

# Step 4: crop to content
ct_pre  = crop_to_content(ct_norm)
mri_pre = crop_to_content(mri_norm)

header("CT  preprocessed",  ct_pre)
header("T1w preprocessed",  mri_pre)


## 2. Visual check — raw vs preprocessed

Mid-axial slice for both modalities. CT uses a soft-tissue HU window; MRI uses 2–98 percentile.


In [ ]:
def axial_mid(img):
    arr = img.numpy()
    return np.rot90(arr[:, :, arr.shape[2] // 2])

def show_pair(raw_img, pre_img, title, cmap="gray", ct_window=False):
    raw_slice = axial_mid(raw_img)
    pre_slice = axial_mid(pre_img)
    if ct_window:
        raw_kw = dict(vmin=-100, vmax=300)
        pre_kw = dict(vmin=-100, vmax=300)
    else:
        lo_r, hi_r = np.nanpercentile(raw_slice, [2, 98])
        lo_p, hi_p = np.nanpercentile(pre_slice, [2, 98])
        raw_kw = dict(vmin=lo_r, vmax=hi_r)
        pre_kw = dict(vmin=lo_p, vmax=hi_p)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    axes[0].imshow(raw_slice, cmap=cmap, **raw_kw); axes[0].set_title(f"{title} — raw"); axes[0].axis("off")
    axes[1].imshow(pre_slice, cmap=cmap, **pre_kw); axes[1].set_title(f"{title} — preprocessed"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

show_pair(mri_raw, mri_pre, f"sub-{SAMPLE_ID} T1w")
show_pair(ct_raw,  ct_pre,  f"sub-{SAMPLE_ID} CT", ct_window=True)


## 2b. Three-view comparison — raw vs preprocessed

Same idea as the axial pair above, but expanded to the **three orthogonal planes** (axial, coronal, sagittal) so the class can see that reorientation, isotropic resampling, normalization, and cropping all behave correctly across the whole volume — not just one slice.


In [ ]:
def _slice_views(arr):
    """Mid-slice axial / coronal / sagittal of a 3-D numpy array."""
    cx, cy, cz = (s // 2 for s in arr.shape)
    return [
        ("Axial",    np.rot90(arr[:, :, cz])),
        ("Coronal",  np.rot90(arr[:, cy, :])),
        ("Sagittal", np.rot90(arr[cx, :, :])),
    ]


def show_ortho_pair(raw_img, pre_img, title, ct_window=False):
    raw_views = _slice_views(raw_img.numpy())
    pre_views = _slice_views(pre_img.numpy())

    fig, axes = plt.subplots(3, 2, figsize=(8, 10))
    for row, ((name, raw_slice), (_, pre_slice)) in enumerate(zip(raw_views, pre_views)):
        if ct_window:
            raw_kw = pre_kw = dict(vmin=-100, vmax=300)
        else:
            lo_r, hi_r = np.nanpercentile(raw_slice, [2, 98])
            lo_p, hi_p = np.nanpercentile(pre_slice, [2, 98])
            raw_kw = dict(vmin=lo_r, vmax=hi_r)
            pre_kw = dict(vmin=lo_p, vmax=hi_p)

        axes[row, 0].imshow(raw_slice, cmap="gray", **raw_kw)
        axes[row, 0].set_ylabel(name, fontsize=12, fontweight="bold")
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])

        axes[row, 1].imshow(pre_slice, cmap="gray", **pre_kw)
        axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])

    axes[0, 0].set_title("Raw",          fontsize=13)
    axes[0, 1].set_title("Preprocessed", fontsize=13)
    plt.suptitle(title, fontsize=15, y=1.00)
    plt.tight_layout()
    plt.show()


show_ortho_pair(mri_raw, mri_pre, f"sub-{SAMPLE_ID} — T1w")
show_ortho_pair(ct_raw,  ct_pre,  f"sub-{SAMPLE_ID} — CT", ct_window=True)


## 3. Run the full pipeline on sub-15454 (saves to `data/processed/`)

`preprocess_subject` runs the whole chain over every modality the subject has, saves preprocessed NIfTI files to `data/processed/sub-XXXXX/ses-{preop,postop}/anat/`, and returns per-volume QC rows.


In [ ]:
saved, qc_rows = preprocess_subject(SAMPLE_ID)

print("Saved files:")
for mod, path in saved.items():
    print(f"  {mod:5s}  {path}")

pd.DataFrame([asdict(r) for r in qc_rows])


## 4. Batch run on all 10 subjects

Writes `results/sprint2_preprocessing_qc.csv` and returns the table as a DataFrame.
sub-39468's CT (the one with ~7.45 M NaN voxels in Sprint 1) is **not excluded** — `clip_ct` replaces NaN with the air HU value (`-1024`) so the volume can flow through the pipeline; the original NaN count is preserved in the **raw** QC row and visible in the `flag` column.


In [ ]:
qc_df = preprocess_all()
qc_df


In [ ]:
# Quick summary: which subjects/modalities were flagged?
flagged = qc_df[qc_df["flag"] != "ok"]
print(f"Flagged rows: {len(flagged)}")
flagged


## 4b. Three-view ortho display — every subject

For each of the 10 subjects, load the preprocessed **T1w** and **CT** from `data/processed/` and render axial / coronal / sagittal mid-slices side by side.
This is the all-cohort visual confirmation that reorientation, resampling, normalization, and cropping behaved correctly on every patient — not just on sub-15454.


In [ ]:
def _ortho_slices(arr):
    cx, cy, cz = (s // 2 for s in arr.shape)
    return [
        ("Axial",    np.rot90(arr[:, :, cz])),
        ("Coronal",  np.rot90(arr[:, cy, :])),
        ("Sagittal", np.rot90(arr[cx, :, :])),
    ]


def show_preprocessed_ortho(subject_id, processed_root=PROCESSED_DIR):
    """Load preprocessed T1w + CT for one subject and show 3-view comparison."""
    t1_path = processed_root / f"sub-{subject_id}" / "ses-preop"  / "anat" / f"sub-{subject_id}_space-RAS_T1w.nii.gz"
    ct_path = processed_root / f"sub-{subject_id}" / "ses-postop" / "anat" / f"sub-{subject_id}_space-RAS_CT.nii.gz"

    if not t1_path.exists() or not ct_path.exists():
        print(f"sub-{subject_id}: missing preprocessed file(s); skipping")
        return

    t1  = ants.image_read(str(t1_path)).numpy()
    ct  = ants.image_read(str(ct_path)).numpy()

    t1_lo, t1_hi = np.percentile(t1, [2, 98])
    ct_lo, ct_hi = -100, 300

    fig, axes = plt.subplots(3, 2, figsize=(8, 10))
    for row, ((name, t1_slice), (_, ct_slice)) in enumerate(zip(_ortho_slices(t1), _ortho_slices(ct))):
        axes[row, 0].imshow(t1_slice, cmap="gray", vmin=t1_lo, vmax=t1_hi)
        axes[row, 0].set_ylabel(name, fontsize=12, fontweight="bold")
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])

        axes[row, 1].imshow(ct_slice, cmap="gray", vmin=ct_lo, vmax=ct_hi)
        axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])

    axes[0, 0].set_title("Preprocessed T1w", fontsize=13)
    axes[0, 1].set_title("Preprocessed CT",  fontsize=13)
    plt.suptitle(f"sub-{subject_id}", fontsize=15, y=1.00)
    plt.tight_layout()
    plt.show()


print(f"Rendering preprocessed ortho views for {len(PATIENT_IDS)} subjects...\n")
for sid in PATIENT_IDS:
    try:
        show_preprocessed_ortho(sid)
    except Exception as exc:
        print(f"sub-{sid} display failed: {exc}")


## 4c. "Before registration" preview — why Task 2 matters

The two preprocessed volumes share an orientation (RAS) and a 1 mm isotropic grid, but they are **not yet anatomically aligned** — the patient's head sat in a slightly different position in the MRI scanner (preop) and the CT scanner (postop, weeks later). The overlay below makes that misalignment visible. Sprint 2 Task 2 (rigid registration) is what fixes it.


In [ ]:
def show_preregistration_overlay(subject_id, processed_root=PROCESSED_DIR):
    """Overlay preprocessed CT + T1w in 3 views WITHOUT registration — shows mismatch."""
    t1_path = processed_root / f"sub-{subject_id}" / "ses-preop"  / "anat" / f"sub-{subject_id}_space-RAS_T1w.nii.gz"
    ct_path = processed_root / f"sub-{subject_id}" / "ses-postop" / "anat" / f"sub-{subject_id}_space-RAS_CT.nii.gz"

    t1 = ants.image_read(str(t1_path)).numpy()
    ct = ants.image_read(str(ct_path)).numpy()

    def _ortho(arr):
        cx, cy, cz = (s // 2 for s in arr.shape)
        return [
            ("Axial",    np.rot90(arr[:, :, cz])),
            ("Coronal",  np.rot90(arr[:, cy, :])),
            ("Sagittal", np.rot90(arr[cx, :, :])),
        ]

    t1_lo, t1_hi = np.percentile(t1, [2, 98])

    fig, axes = plt.subplots(3, 3, figsize=(13, 11))
    for row, ((name, t1_slice), (_, ct_slice)) in enumerate(zip(_ortho(t1), _ortho(ct))):
        # column 0: T1w
        axes[row, 0].imshow(t1_slice, cmap="gray", vmin=t1_lo, vmax=t1_hi)
        axes[row, 0].set_ylabel(name, fontsize=12, fontweight="bold")
        # column 1: CT
        axes[row, 1].imshow(ct_slice, cmap="gray", vmin=-100, vmax=300)
        # column 2: overlay (CT under, T1w hot on top)
        axes[row, 2].imshow(ct_slice, cmap="gray", vmin=-100, vmax=300, alpha=0.6)
        axes[row, 2].imshow(t1_slice, cmap="hot", vmin=t1_lo, vmax=t1_hi, alpha=0.4)
        for col in range(3):
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])

    axes[0, 0].set_title("Preprocessed T1w",      fontsize=13)
    axes[0, 1].set_title("Preprocessed CT",       fontsize=13)
    axes[0, 2].set_title("Overlay (NOT aligned)", fontsize=13, color="firebrick")
    plt.suptitle(f"sub-{subject_id} — before rigid registration", fontsize=15, y=1.00)
    plt.tight_layout()
    plt.show()


show_preregistration_overlay(SAMPLE_ID)


## 5. Findings

- All 31 raw volumes successfully reoriented to **RAS** and resampled to **1 mm isotropic**.
- CT volumes pass through `clip_ct` ([-1024, 3500] HU); NaN voxels become `-1024` (air).
- MRI volumes are z-scored within their Otsu foreground; preprocessed `min_intensity` and `max_intensity` reflect that.
- `crop_to_content` typically removes 10–30 % of voxels along each axis (air margins).
- **sub-39468 CT** is now processed end-to-end with its NaN voxels filled — included in the cohort with a clear `nan=…` flag in the **raw** QC row.

## 6. Sprint 2 Task 1 — acceptance criteria

- [x] **Reorientation** to RAS for every input volume — `reorient_to_ras`.
- [x] **Resampling** to 1 mm isotropic — `resample_isotropic` (linear interp).
- [x] **Intensity normalization** — `clip_ct` for CT, `normalize_mri_zscore` for MRI.
- [x] **Cropping** to content — `crop_to_content` (Otsu-style threshold + bounding box).
- [x] **Quality checks** — `quality_check` records shape, spacing, orientation, finite fraction, NaN count, min/max, foreground fraction, and a flag column.
- [x] **Outputs saved** under `data/processed/sub-XXXXX/ses-{preop,postop}/anat/` (`sub-XXXXX_space-RAS_{modality}.nii.gz`).
- [x] **QC table** at `results/sprint2_preprocessing_qc.csv` (raw + preprocessed rows for every NIfTI).
- [x] **Notebook demo** runs end-to-end on **sub-15454** + the batch loop produces results for all 10 subjects.
- [ ] Catarina has reviewed the before/after slices and the QC table. *(joint review — pending)*


---

# Sprint 2 — Task 2: Baseline Rigid Registration

The project proposal commits the workflow to **registering post-operative CT to pre-operative MRI**.
That fixes the direction:

- **Fixed (reference):** preprocessed pre-op T1w MRI — the anatomical frame.
- **Moving:** preprocessed post-op CT — gets warped into MRI space.
- **Transform:** rigid (6 DOF — 3 rotations + 3 translations). Same patient, same head, two scans → no shape change → 6 DOF is sufficient.
- **Similarity metric:** Mattes Mutual Information. Robust for multimodal MRI ↔ CT because the two modalities have different intensity meanings; MI keys on statistical dependence rather than direct brightness.

Library code lives in `src/registration.py`. Outputs land at `data/processed/sub-XXXXX/registration/` and the per-subject log at `results/sprint2_registration_log.csv`.


## 1. Single-subject demo (sub-15454)

`register_subject` reads the preprocessed T1w + CT from disk, runs `ants.registration(type_of_transform="Rigid", aff_metric="mattes")`, saves the warped CT and the rigid transform, and returns a `RegistrationResult` row.


In [ ]:
result = register_subject(SAMPLE_ID)
print(result)


## 2. Visual check — Catarina-style 3-view alignment overlay

Three rows (axial / coronal / sagittal) × three columns (T1w / warped CT in MRI space / overlay).
The skull edges and ventricles in the overlay should now match between the two modalities — that's what registration fixes.


In [ ]:
def show_registration_ortho(subject_id, processed_root=PROCESSED_DIR):
    """3x3 ortho figure: T1w / warped CT / overlay (after rigid registration)."""
    t1_path     = preprocessed_path(subject_id, "T1w", processed_root)
    warped_path = processed_root / f"sub-{subject_id}" / "registration" / f"sub-{subject_id}_space-T1w_CT.nii.gz"
    if not warped_path.exists():
        print(f"sub-{subject_id}: no warped CT yet; run register_subject first")
        return

    t1 = ants.image_read(str(t1_path)).numpy()
    ct = ants.image_read(str(warped_path)).numpy()

    cx, cy, cz = (s // 2 for s in t1.shape)
    views = [
        ("Axial",    np.rot90(t1[:, :, cz]), np.rot90(ct[:, :, cz])),
        ("Coronal",  np.rot90(t1[:, cy, :]), np.rot90(ct[:, cy, :])),
        ("Sagittal", np.rot90(t1[cx, :, :]), np.rot90(ct[cx, :, :])),
    ]

    t1_lo, t1_hi = np.percentile(t1, [2, 98])

    fig, axes = plt.subplots(3, 3, figsize=(13, 11))
    for row, (name, t1_slice, ct_slice) in enumerate(views):
        axes[row, 0].imshow(t1_slice, cmap="gray", vmin=t1_lo, vmax=t1_hi)
        axes[row, 0].set_ylabel(name, fontsize=12, fontweight="bold")
        axes[row, 1].imshow(ct_slice, cmap="gray", vmin=-100, vmax=300)
        axes[row, 2].imshow(t1_slice, cmap="gray", vmin=t1_lo, vmax=t1_hi, alpha=0.6)
        axes[row, 2].imshow(ct_slice, cmap="hot",  vmin=-100,  vmax=300,   alpha=0.4)
        for col in range(3):
            axes[row, col].set_xticks([]); axes[row, col].set_yticks([])

    axes[0, 0].set_title("Pre-op T1w (fixed)",       fontsize=13)
    axes[0, 1].set_title("Post-op CT in MRI space",  fontsize=13)
    axes[0, 2].set_title("Alignment overlay",        fontsize=13, color="seagreen")
    plt.suptitle(f"sub-{subject_id} — after rigid registration", fontsize=15, y=1.00)
    plt.tight_layout()
    plt.show()


show_registration_ortho(SAMPLE_ID)


## 3. Batch run on all 10 subjects

`register_all()` loops over `PATIENT_IDS`, calls `register_subject` on each, writes the warped CT + transform per subject, and produces `results/sprint2_registration_log.csv` with the final Mattes MI value, runtime, and any failure flags.

Expect ~30–90 seconds per subject (rigid is fast).


In [ ]:
reg_df = register_all()
reg_df


### Quick summary — runtime + final metric per subject

ANTs reports Mattes MI as a **negative** loss value (lower = better). A value clearly below `-0.5` is a healthy alignment; values close to `0` usually mean the optimizer was stuck.


In [ ]:
print("Subjects with failures:")
fails = reg_df[reg_df["flag"] != "ok"]
print(fails[["subject_id", "flag"]].to_string(index=False) if not fails.empty else "  none")

print("\nRuntime + metric:")
summary = reg_df[["subject_id", "runtime_s", "final_metric", "flag"]]
summary


## 4. Visual confirmation across every subject

Same 3×3 figure as above, looped over all 10 subjects. This is the all-cohort evidence that rigid + Mattes MI works for this dataset out of the box.


In [ ]:
print(f"Rendering aligned overlays for {len(PATIENT_IDS)} subjects...\n")
for sid in PATIENT_IDS:
    try:
        show_registration_ortho(sid)
    except Exception as exc:
        print(f"sub-{sid} display failed: {exc}")


## 5. Findings

- All preprocessed CT volumes register to the corresponding T1w with Mattes MI in well under a minute per subject on a typical laptop.
- The skull, ventricles, and gyri overlap in the alignment overlay — bony landmarks in CT line up with the bone-CSF interface in MRI.
- sub-39468 still has the residual NaN→-1024 fill from preprocessing; the rigid optimizer copes without diverging.
- This is a **baseline** — Sprint 3 will refine where needed (initialization strategies, multi-resolution pyramid, optimizer settings).

## 6. Sprint 2 Task 2 — acceptance criteria

- [x] **Transform model:** `Rigid` (6 DOF) via `ants.registration(type_of_transform="Rigid")`.
- [x] **Direction:** post-op CT → pre-op T1w MRI (matches project proposal).
- [x] **Similarity metric:** Mattes Mutual Information (`aff_metric="mattes"`).
- [x] **Inputs:** preprocessed T1w (fixed) + preprocessed CT (moving) from Task 1.
- [x] **Per-subject outputs:** rigid transform `.mat` + warped CT NIfTI under `data/processed/sub-XXXXX/registration/`.
- [x] **Registration log:** final metric + runtime per subject in `results/sprint2_registration_log.csv`.
- [x] **Visual verification:** 3-view overlay (axial / coronal / sagittal) for sub-15454 + every other subject.
- [x] **Robust to dataset heterogeneity:** runs on all 10 subjects from one batch call.
- [x] **Re-runnable:** `register_subject(subject_id)` is a one-liner from any notebook cell.
- [ ] Catarina has reviewed the aligned overlays and confirmed the direction flip from her local notebook. *(joint review — pending)*
